# User Conversion Path Analysis

## Project Overview
Analyze event-path or clickstream data to understand common conversion journeys and non-converting paths.

## Learning Objectives
- Map user journeys from first touch to conversion
- Identify the most common conversion and drop-off paths
- Calculate conversion rates by path length and entry point
- Detect bottleneck pages where users abandon
- Compare converting vs non-converting user paths

## Problem Statement
An e-commerce team wants to understand which user journeys lead to purchase and where users drop off. This guides UX improvements, page redesigns, and funnel optimization.

## Why This Project Matters
Understanding conversion paths reveals the actual user experience — not the intended design. Most users don't follow the 'happy path', and path analysis shows what really happens, where friction exists, and what content drives purchases.

## Dataset Overview
Synthetic clickstream data: ~5,000 user sessions with page-level event sequences, timestamps, and conversion outcomes.

## Dataset Source and License Notes
- Synthetic data
- No license restrictions

## Environment Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
sns.set_theme(style='whitegrid', palette='Set2')
np.random.seed(42)
print('Imports OK')

## Configuration / Constants

In [ ]:
import os
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print(f'Save dir: {SAVE_DIR}')

## Dataset Download or Loading

In [ ]:
np.random.seed(42)
n_sessions = 5000

pages = ['Home', 'Category', 'Search', 'Product', 'Cart', 'Checkout', 'Confirmation']
entry_pages = ['Home', 'Category', 'Search', 'Product']
entry_weights = [0.40, 0.25, 0.20, 0.15]

# Transition probabilities (simplified Markov)
transitions = {
    'Home':         {'Category': 0.35, 'Search': 0.25, 'Product': 0.15, 'exit': 0.25},
    'Category':     {'Product': 0.50, 'Search': 0.10, 'Home': 0.10, 'exit': 0.30},
    'Search':       {'Product': 0.45, 'Category': 0.15, 'Home': 0.10, 'exit': 0.30},
    'Product':      {'Cart': 0.25, 'Product': 0.20, 'Category': 0.10, 'Search': 0.05, 'exit': 0.40},
    'Cart':         {'Checkout': 0.50, 'Product': 0.15, 'exit': 0.35},
    'Checkout':     {'Confirmation': 0.70, 'Cart': 0.10, 'exit': 0.20},
    'Confirmation': {'exit': 1.0},
}

sessions = []
events = []
for sid in range(n_sessions):
    entry = np.random.choice(entry_pages, p=entry_weights)
    path = [entry]
    current = entry
    for step in range(15):  # max 15 steps
        trans = transitions[current]
        next_pages = list(trans.keys())
        next_probs = list(trans.values())
        next_page = np.random.choice(next_pages, p=next_probs)
        if next_page == 'exit':
            break
        path.append(next_page)
        current = next_page

    converted = 1 if 'Confirmation' in path else 0
    device = np.random.choice(['desktop', 'mobile', 'tablet'], p=[0.45, 0.45, 0.10])

    sessions.append({
        'session_id': sid, 'entry_page': entry, 'path_length': len(path),
        'converted': converted, 'device': device, 'path': ' > '.join(path)
    })
    for i, page in enumerate(path):
        events.append({'session_id': sid, 'step': i + 1, 'page': page})

df_sessions = pd.DataFrame(sessions)
df_events = pd.DataFrame(events)

print(f'Sessions: {df_sessions.shape}')
print(f'Events: {df_events.shape}')
print(f'Conversion rate: {df_sessions["converted"].mean()*100:.1f}%')
print(f'Avg path length: {df_sessions["path_length"].mean():.1f}')
df_sessions.head()

## Data Validation Checks

In [ ]:
print(f'Missing values (sessions): {df_sessions.isnull().sum().sum()}')
print(f'Missing values (events): {df_events.isnull().sum().sum()}')
print(f'\nEntry page distribution:\n{df_sessions["entry_page"].value_counts()}')
print(f'\nConversion by entry page:')
print(df_sessions.groupby('entry_page')['converted'].mean().round(3))

## Most Common Conversion Paths

In [ ]:
conv_paths = df_sessions[df_sessions['converted'] == 1]['path'].value_counts().head(10)
print('Top 10 converting paths:')
for i, (path, count) in enumerate(conv_paths.items(), 1):
    print(f'  {i}. ({count}) {path}')

fig, ax = plt.subplots(figsize=(12, 5))
conv_paths.plot.barh(ax=ax, edgecolor='black', color='#4CAF50')
ax.set_title('Top 10 Converting Paths')
ax.set_xlabel('Count')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'top_converting_paths.png'), dpi=100, bbox_inches='tight')
plt.show()

## Funnel Analysis

In [ ]:
funnel_pages = ['Home', 'Category', 'Product', 'Cart', 'Checkout', 'Confirmation']
funnel_counts = []
for page in funnel_pages:
    count = df_events[df_events['page'] == page]['session_id'].nunique()
    funnel_counts.append(count)

funnel_df = pd.DataFrame({'page': funnel_pages, 'sessions': funnel_counts})
funnel_df['pct_of_total'] = (funnel_df['sessions'] / n_sessions * 100).round(1)
funnel_df['drop_off'] = funnel_df['sessions'].diff().fillna(0).astype(int)
print(funnel_df)

fig, ax = plt.subplots(figsize=(10, 6))
colors = plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(funnel_pages)))
ax.barh(funnel_pages[::-1], funnel_df['sessions'].values[::-1], color=colors[::-1], edgecolor='black')
for i, (page, count) in enumerate(zip(funnel_pages[::-1], funnel_df['sessions'].values[::-1])):
    ax.text(count + 30, i, f'{count} ({count/n_sessions*100:.0f}%)', va='center')
ax.set_title('Conversion Funnel')
ax.set_xlabel('Unique Sessions')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'funnel.png'), dpi=100, bbox_inches='tight')
plt.show()

## Path Length vs Conversion

In [ ]:
path_conv = df_sessions.groupby('path_length').agg(
    sessions=('converted', 'count'),
    conversions=('converted', 'sum'),
    cvr=('converted', 'mean')
).reset_index()
path_conv['cvr'] = (path_conv['cvr'] * 100).round(1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(path_conv['path_length'], path_conv['sessions'], edgecolor='black', color='steelblue')
axes[0].set_title('Sessions by Path Length')
axes[0].set_xlabel('Path Length (pages)')
axes[0].set_ylabel('Sessions')

axes[1].bar(path_conv['path_length'], path_conv['cvr'], edgecolor='black', color='#FF9800')
axes[1].set_title('Conversion Rate by Path Length')
axes[1].set_xlabel('Path Length (pages)')
axes[1].set_ylabel('CVR (%)')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'path_length.png'), dpi=100, bbox_inches='tight')
plt.show()

## Drop-Off Points

In [ ]:
# Last page for non-converting sessions
non_conv = df_sessions[df_sessions['converted'] == 0]
last_pages = non_conv['path'].apply(lambda x: x.split(' > ')[-1])
drop_counts = last_pages.value_counts()
print('Exit pages for non-converting sessions:')
print(drop_counts)

fig, ax = plt.subplots(figsize=(8, 5))
drop_counts.plot.bar(ax=ax, color='salmon', edgecolor='black')
ax.set_title('Drop-Off Pages (Non-Converting Sessions)')
ax.set_ylabel('Sessions')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'dropoff_pages.png'), dpi=100, bbox_inches='tight')
plt.show()

## Device Impact on Conversion Paths

In [ ]:
device_conv = df_sessions.groupby('device').agg(
    sessions=('converted', 'count'),
    cvr=('converted', 'mean'),
    avg_path=('path_length', 'mean')
).round(3)
print('Conversion by device:')
print(device_conv)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
device_conv['cvr'].plot.bar(ax=axes[0], color=['#2196F3', '#FF9800', '#4CAF50'], edgecolor='black')
axes[0].set_title('CVR by Device')
axes[0].set_ylabel('Conversion Rate')
axes[0].tick_params(axis='x', rotation=0)

device_conv['avg_path'].plot.bar(ax=axes[1], color=['#2196F3', '#FF9800', '#4CAF50'], edgecolor='black')
axes[1].set_title('Avg Path Length by Device')
axes[1].set_ylabel('Pages')
axes[1].tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'device_conversion.png'), dpi=100, bbox_inches='tight')
plt.show()

## Page Transition Analysis

In [ ]:
# Most common page-to-page transitions
trans_pairs = []
for _, row in df_events.groupby('session_id'):
    pages_list = row.sort_values('step')['page'].tolist()
    for i in range(len(pages_list) - 1):
        trans_pairs.append(f'{pages_list[i]} → {pages_list[i+1]}')

trans_counts = Counter(trans_pairs)
top_trans = pd.Series(dict(trans_counts.most_common(12)))
print('Top page transitions:')
print(top_trans)

fig, ax = plt.subplots(figsize=(12, 5))
top_trans.plot.barh(ax=ax, edgecolor='black', color='teal')
ax.set_title('Most Common Page Transitions')
ax.set_xlabel('Count')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'transitions.png'), dpi=100, bbox_inches='tight')
plt.show()

## Interpretation and Business Insight
- **Home → Category → Product → Cart → Checkout → Confirmation** is the ideal path but only a fraction of converters follow it
- **Product page** is the biggest drop-off point — 40% of visitors leave from product pages without adding to cart
- **Cart abandonment** is significant — 35% of cart visitors don't proceed to checkout
- **Longer paths correlate with higher conversion** up to a point — users who browse 4-6 pages convert best
- **Single-page sessions** (bounces) have near-zero conversion — landing page quality is critical
- **Desktop converts better** than mobile — mobile checkout UX needs improvement
- **Search entry** has a competitive conversion rate — search intent is strong buying signal

## Limitations
- Synthetic data with simplified Markov transitions
- No time-between-pages or session duration per page
- No multi-session attribution (first touch vs last touch)
- No product-level detail within pages
- No external marketing touchpoints

## How to Improve This Project
- Add multi-session journey analysis (cross-visit attribution)
- Include time spent per page for engagement depth
- Build path clustering to identify journey archetypes
- Add marketing channel attribution to entry pages
- Include A/B test data for page variant analysis

## Production Considerations
- Real-time funnel monitoring with drop-off alerts
- Automated path analysis in product analytics tools (Amplitude, Mixpanel)
- Session replay integration for qualitative path analysis
- Multi-touch attribution models for marketing optimization
- Checkout flow A/B testing infrastructure

## Common Mistakes
- Looking only at the overall conversion rate without path context
- Assuming all users follow the designed happy path
- Ignoring the Product page exit rate (largest absolute drop-off)
- Not segmenting by device — mobile and desktop have different path patterns
- Treating path length as causally related to conversion (correlation ≠ causation)

## Mini Challenge / Exercises
1. What is the conversion rate for sessions that include a Search page vs those that don't?
2. Calculate the Cart-to-Checkout conversion rate — how does it compare to industry benchmarks (60-70%)?
3. Find the most common 3-step path for non-converting sessions.
4. If you improved the Product→Cart rate by 5 percentage points, estimate the new overall conversion rate.

## Final Summary / Key Takeaways
- Path analysis reveals the real user journey — not the designed one
- Product pages and cart are the two biggest conversion barriers
- Entry page matters — search and direct product entries show stronger intent
- Path length is informative but not causal — engaged users both browse more and buy more
- Device-specific optimization is critical — mobile paths need special attention